In [1]:
# -*- coding: utf-8 -*-
"""
Sentiment Analysis & Thematic Clustering
on a single “Reflective Interview” transcript (Participant #72),
printing every segment per topic.
"""

# 0) Monkey-patch to avoid UMAP → TensorFlow recursion errors
import sys, types
sys.modules['umap.parametric_umap'] = types.ModuleType('umap.parametric_umap')

# 1) Patch inspect.ArgSpec for Python 3.12+
import inspect
from collections import namedtuple
if not hasattr(inspect, "ArgSpec"):
    inspect.ArgSpec = namedtuple("ArgSpec", ["args","varargs","keywords","defaults"])

# 2) Imports & dependencies (install once):
#    pip install transformers torch scikit-learn bertopic umap-learn hdbscan
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from collections import Counter, defaultdict

# 3) Raw interview text
interview_text = """
P 39:Main activities very low due client base. Somebody participates meetings City Mission over project. Few clients impacted, mainly because can't followed past discharge due CFR HIPAA regulations chemical abuse clients. Once gone, not followed.
P 60:Included looking Healthy Link alerts, informing providers about transition, alerts informing transitional case managers Jericho Road, Health Home Care Coordinator team about transitions, trying work together provide best care somebody transitioning hospital, ensuring good care outpatient setting moving forward. Going: currently planning, discussion stage. Great discussions. Hopeful start getting alerts, working collaboratively moving forward.
P 88:Positive things. Development clinical team, Advisory Board going well. Participants interested doing something different. Don’t realize how different. Trying hard adapt past practices work here. Easier someone without previous pattern. Clinician never Buffalo City Mission prior, extremely flexible, willing try different things. Others long-time patterns harder change. Push, pull dynamics. Pleased where now. Uncertainty reimbursement respite service. Physician services reimbursed, behavioral health reimbursed — how pay bed, 24/7 supervision? Different city mission others. Ongoing problem, common all units. City Mission trying make work, planning pilot reduced rate, develop data demonstrate project effectiveness.
P 94:Work group good. View medical respite, consider pros, cons, prevent issues. Aim smooth, collaborative relationship HEALTH SYSTEM, City Mission. Disconnect Advisory Board, City Mission, hospital, contract, us. Negotiations ongoing. HEALTH SYSTEM hadn’t signed contract. Educating group unsure participation. Leadership agreed participation. Transparency lacking. Tipping point: emergency rooms holding 50 patients, placement issues. Prompted urgency. Transparency needed work group, status.
P 69:Current activities: logistics, identifying data pieces, providers, integration, delivery. Research conversations — ideal vs. reality capability. On way good product research, alerts. Beneficial both sides. Expecting good data, good products Sharon, team. Advancing notifications. So far, so good. Currently: product logistics.
P 26:Main activities: reports generated Jericho Road, Sharon's help — high needs report. Subscribe, notify list. HEALTHeLINK side: ingest reports database, algorithm (Sharon’s/HEL’s side) assembles useful patient data enhanced care alerts. Database populated → HEL builds channel (interface engine) → queries database → pulls fields → creates alert → sends direct mail Jericho. Trigger: ADT message Jericho Road patient. If seen hospital → HEL receives ADT → queries tables → sends enhanced alert.Going okay. Rough start. Chaotic project management, unclear responsibilities. Regrouped. Clear path forward. Feeling good now.
P 91:Enhanced care alert main focus. About released. Going: well, fine. Complicated moving pieces. Good progress.
P 84:Jericho Road understaffed. Worker could catch up working half time (20–30 hours/week). Allowed couple days/week only. Never caught up. Personal input: 12 hours/week. Ongoing process. Patients constantly discharged. Restart list every 3 days. Missed ones skipped. Sometimes 4–5 days out.
P 99:Current activity: building expanded high needs report based initial, additional criteria (Dr. ABC). Rolling out. Final project ready. Project coming together after year+. Dr. ABC great, kind. Even mistakes addressed kindly — suggests iterations nicely. Nice seeing implementation near.
P 72:Interesting. Don’t fully understand process. Working group deciding things → presented advisory committee. Made changes organization based second review. Project led CLINICAL SCHOLAR different other City Mission projects. Things mix up sometimes.
P 79:Phase: deploying care alert. Based yesterday’s meeting — wheels falling off. Impressions: behavioral health provider has few relevant clients, Jericho Road not ready. Holding pattern. Observing developments.

"""

# 4) Extract only participant *sentences* into `docs`
import re

# Split the entire transcript on full stops
sentences = re.split(r'\.\s*', interview_text)

docs = []
for sent in sentences:
    sent = sent.strip()
    if not sent:
        continue
    # Skip questions, interviewer prompts, headers
    if sent.startswith(("Q", "Interviewer", "Reflective Interview",
                        "Participant ID", "Time Point")):
        continue
    # Remove leading "P <number>:" tag
    sent = re.sub(r'^P\s*\d+\s*:\s*', '', sent)
    docs.append(sent)

# 5) Clean out filler tokens (um/uh)
docs = [re.sub(r"\b(um|uh+)\b", "", seg, flags=re.IGNORECASE).strip()
        for seg in docs]


# 6) Load ClinicalBERT & define embed()
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model     = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").eval()

def embed(text: str) -> np.ndarray:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        last_hidden = model(**inputs).last_hidden_state
    return last_hidden[:,0].cpu().numpy()[0]

# 7) Build sentiment prototypes & classify
prototypes = {
    "positive": embed("I’m very pleased with how it’s going."),
    "negative": embed("I’m frustrated and unhappy with this."),
    "neutral":  embed("It’s neither good nor bad; it’s just okay.")
}

def classify_sentiment(seg: str) -> str:
    v    = embed(seg)
    sims = {lbl: cosine_similarity([v],[p])[0,0] for lbl,p in prototypes.items()}
    return max(sims, key=sims.get)

sentiments = [classify_sentiment(d) for d in docs]

print("\n--- Sentiment Analysis ---")
for seg, s in zip(docs, sentiments):
    print(f"[{s.upper():7}] {seg}")

# 8) Build & normalize embeddings
embs = np.stack([embed(d) for d in docs])
embs = normalize(embs)

# 9) Configure UMAP (random init) & HDBSCAN for BERTopic
umap_model = UMAP(n_neighbors=5, n_components=2, min_dist=0.1,
                  metric="cosine", init="random")
hdbscan_model = HDBSCAN(min_cluster_size=2, min_samples=1,
                        metric="euclidean", cluster_selection_method="eom")

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=None,
    verbose=False
)

# 10) Fit & then PRINT ALL segments per topic
topics, probs = topic_model.fit_transform(docs, embs)

print("\n--- Document Count per Topic ---")
print(Counter(topics))

info = topic_model.get_topic_info()
print("\n--- Topic Info ---")
print(info)

print("\n--- Top Keywords per Topic ---")
for tid in info.Topic.unique():
    if tid == -1: continue
    words = topic_model.get_topic(tid)
    print(f"Topic {tid}:", [w for w,_ in words][:10])

print("\n--- All Segments per Topic ---")
docs_arr = np.array(docs)
by_topic = defaultdict(list)
for i, t in enumerate(topics):
    by_topic[t].append(i)

for tid, idxs in by_topic.items():
    label = "Noise" if tid == -1 else f"Topic {tid}"
    print(f"\n-- {label} ({len(idxs)} segments) --")
    for i in idxs:
        print(" •", docs_arr[i])
        
with open("topic_analysisPID69_HeatheLink.txt","w", encoding="utf-8") as f:
    for tid, idxs in by_topic.items():
        label = "Noise" if tid == -1 else f"Topic {tid}"
        f.write(f"\n-- {label} ({len(idxs)} segments) --\n")
        for i in idxs:
            f.write(f" • {docs_arr[i]}\n")
print("Wrote full analysis to topic_analysis.txt")



--- Sentiment Analysis ---
[NEUTRAL] Main activities very low due client base
[POSITIVE] Somebody participates meetings City Mission over project
[NEUTRAL] Few clients impacted, mainly because can't followed past discharge due CFR HIPAA regulations chemical abuse clients
[NEUTRAL] Once gone, not followed
[NEUTRAL] Included looking Healthy Link alerts, informing providers about transition, alerts informing transitional case managers Jericho Road, Health Home Care Coordinator team about transitions, trying work together provide best care somebody transitioning hospital, ensuring good care outpatient setting moving forward
[NEGATIVE] Going: currently planning, discussion stage
[NEGATIVE] Great discussions
[POSITIVE] Hopeful start getting alerts, working collaboratively moving forward
[NEUTRAL] Positive things
[POSITIVE] Development clinical team, Advisory Board going well
[POSITIVE] Participants interested doing something different
[POSITIVE] Don’t realize how different
[NEUTRAL] Trying 

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



--- Document Count per Topic ---
Counter({1: 5, 0: 5, 2: 5, 7: 4, 3: 4, 4: 4, 6: 4, 5: 4, 16: 3, 11: 3, 15: 3, 8: 3, 18: 3, 17: 3, 19: 3, 14: 3, 9: 3, 13: 3, -1: 3, 12: 3, 10: 3, 23: 2, 26: 2, 25: 2, 24: 2, 20: 2, 22: 2, 21: 2, 27: 2})

--- Topic Info ---
    Topic  Count                                           Name  \
0      -1      3            -1_unsure_educating_product_process   
1       0      5          0_clear_notifications_final_advancing   
2       1      5                      1_flexible_only_gone_once   
3       2      5                     2_rough_push_pull_dynamics   
4       3      4         3_personal_reimbursement_message_hadnt   
5       4      4                          4_rolling_on_kind_way   
6       5      4                  5_database_queries_sends_side   
7       6      4                        6_so_good_progress_okay   
8       7      4                  7_informing_alerts_care_about   
9       8      3        8_urgency_prompted_positive_interesting   
10    

In [ ]:


# 1) map to numeric
score_map = {"negative": -1, "neutral": 0, "positive": +1}
scores    = [ score_map[s] for s in sentiments ]

# 2) overall average
avg_score = sum(scores) / len(scores)

# 3) also get a simple distribution
from collections import Counter
dist = Counter(sentiments)

print("Average sentiment score:", avg_score)
print("Distribution:", dist)


Average sentiment score: 0.17777777777777778
Distribution: Counter({'neutral': 52, 'positive': 27, 'negative': 11})


In [3]:
total = len(sentiments)
pct_pos = dist["positive"] / total * 100
pct_neg = dist["negative"] / total * 100
pct_nat = dist["neutral"]  / total * 100

print(f"Positive: {pct_pos:.1f}%, Neutral: {pct_nat:.1f}%, Negative: {pct_neg:.1f}%")


Positive: 30.0%, Neutral: 57.8%, Negative: 12.2%
